**Note for reviewers.** This notebook also contains analyses that are not reported in the paper: partner-swap decision flips. Player-removal flips are reported. Those sections are kept only so the notebook runs as one unit, and their outputs should not be read as paper results.

# New wd=0.1 benchmark follow-up (execution preparation)
Exploratory validation analysis; historical saved outputs cleared.
Historical numerical prose is background, not a result of this benchmark.
Output-key scrubbing does not establish a serial algorithm. MB16 legacy scrub results
are invalidated by F25; use Section R corrected results. No new test access is permitted.


# MB18 — Where Is the Decision Made, Is There Backup, and Does Another Model Do the Same?

**Track B · Follow-up to MB06–MB16 · Identity × Context project**

| Item | Content |
|---|---|
| **Questions** | (N9) When removing the player or changing the context **changes the model's chosen action**, which component restores the original choice, and which one alone produces the change? (N8) At which layer is the choice fixed: gradually, or in one layer? (N7) When one MLP is removed, do later layers make up for it (self-repair), and do pairs of MLPs back each other up (redundancy)? (N10) Do the other four SEL-4L seeds and the two reference models REF-2L and REF-4L show the same site profile and the same behaviour (C1–C4)? |
| **Why** | MB09 showed which MLP *writes* most of Δ (L3), but the component that writes the most is not necessarily where the decision is made. Patching that tracks the chosen action, layer-wise decision points and direct-vs-total effects separate "writes" from "decides". Replication on other models is the cheapest test of generality, since the checkpoints already exist. |
| **Population** | P-player validation with a context donor and a matched partner. Estimation half for every primary number. |
| **Status** | Follow-up after the deep MI audit; exploratory; not part of the MB12 lock. The plan is written and fingerprinted before any number. No test data. |
| **Outputs** | `config/MB18_plan.json`, `results/MB18_*`, `figures/MB18_*` |

In [1]:
import os
import sys
import json
import time
import importlib
import importlib.util
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")


def find_root(start):
    """`start` (or one of its ancestors) holding both project folders directly, else None."""
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "model_training_evaluation").is_dir() and (candidate / "mi_analysis").is_dir():
            return candidate
    return None


def search_root(bases, max_depth=6):
    """Breadth-first search under each base (e.g. a mounted Drive) for the project root."""
    for base in bases:
        base = Path(base)
        if not base.is_dir():
            continue
        found = find_root(base)
        if found is not None:
            return found
        frontier = [(base, 0)]
        while frontier:
            folder, depth = frontier.pop(0)
            if depth >= max_depth:
                continue
            try:
                children = [c for c in folder.iterdir() if c.is_dir() and not c.name.startswith(".")]
            except PermissionError:
                continue
            for child in children:
                if child.name == "model_training_evaluation" and (child.parent / "mi_analysis").is_dir():
                    return child.parent
                frontier.append((child, depth + 1))
    return None


if "SOCCER_MI_ROOT" in os.environ:
    ROOT = Path(os.environ["SOCCER_MI_ROOT"])
elif IN_COLAB:
    ROOT = search_root(["/content", "/content/drive/MyDrive", "/content/drive/My Drive"])
    if ROOT is None:
        raise RuntimeError(
            "BLOCKED: could not find a folder holding both model_training_evaluation/ and mi_analysis/ "
            "under /content or the mounted Drive. Upload or sync the project there, or set "
            "SOCCER_MI_ROOT to its exact path in a cell ABOVE this one, for example: "
            "import os; os.environ['SOCCER_MI_ROOT'] = '/content/drive/MyDrive/.../soccer_mi'")
else:
    ROOT = find_root(Path.cwd())
    if ROOT is None:
        raise RuntimeError(f"BLOCKED: no ancestor of {Path.cwd()} holds both model_training_evaluation/ "
                           "and mi_analysis/. Run this notebook from inside the project, or set SOCCER_MI_ROOT.")
TRACK_A_DIR = ROOT / "model_training_evaluation"
MI_DIR = Path(os.environ.get("MI_ANALYSIS_DIR", ROOT / "mi_analysis")).resolve()
assert (TRACK_A_DIR / "src" / "data.py").exists(), (
    f"BLOCKED: {TRACK_A_DIR / 'src' / 'data.py'} not found -- ROOT resolved to {ROOT}, which is wrong. "
    "Set SOCCER_MI_ROOT explicitly.")
assert (TRACK_A_DIR / "artifacts" / "events_prepared.parquet").exists(), (
    f"BLOCKED: {TRACK_A_DIR / 'artifacts' / 'events_prepared.parquet'} not found -- ROOT is wrong.")
RUN_MODE = os.environ.get("MI_RUN_MODE", "full")
assert RUN_MODE in ("full", "smoke")

for folder in [MI_DIR, MI_DIR / "src", MI_DIR / "notebooks", MI_DIR / "tests"]:
    folder.mkdir(parents=True, exist_ok=True)
(MI_DIR / "src" / "__init__.py").touch()
os.chdir(MI_DIR)

sys.path[:] = [entry for entry in sys.path if entry not in ("", ".", str(MI_DIR))]
if str(TRACK_A_DIR) not in sys.path:
    sys.path.insert(0, str(TRACK_A_DIR))
spec = importlib.util.spec_from_file_location("mi_src", MI_DIR / "src" / "__init__.py",
                                              submodule_search_locations=[str(MI_DIR / "src")])
mi_src = importlib.util.module_from_spec(spec)
sys.modules["mi_src"] = mi_src
spec.loader.exec_module(mi_src)

print("Colab:", IN_COLAB, "| RUN_MODE:", RUN_MODE)
print("ROOT:", ROOT)
print("Track A (read only):", TRACK_A_DIR, "| exists:", TRACK_A_DIR.exists())
print("Track B:", MI_DIR)

Colab: False | RUN_MODE: full
ROOT: /content/soccer_mi
Track A (read only): /content/soccer_mi/model_training_evaluation | exists: True
Track B: /content/soccer_mi/.mi_analysis_rerun


In [2]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import src.data as data
import src.metrics as metrics
import src.training as training
import mi_src.paths as paths
import mi_src.fingerprint as fingerprint
import mi_src.mi_setup as mi_setup
import mi_src.hooks as hooks
import mi_src.interventions as interventions
import mi_src.mi_metrics as mi_metrics
import mi_src.patching as patching
import mi_src.circuits as circuits
for module in [data, metrics, training, paths, fingerprint, mi_setup, hooks, interventions, mi_metrics, patching, circuits]:
    importlib.reload(module)

if getattr(paths, "SCHEMA_VERSION", 1) < 2:
    raise RuntimeError("BLOCKED: mi_analysis/src/paths.py is older than this notebook expects. "
                       "Re-run MB01 section 0 (its three %%writefile cells), then restart the kernel.")
paths.check_modules(**{name: module for name, module in
                       [("paths", paths), ("fingerprint", fingerprint), ("mi_setup", mi_setup),
                        ("patching", globals().get("patching")), ("circuits", globals().get("circuits"))]
                       if module is not None})

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

OUT = paths.output_dir(MI_DIR, RUN_MODE)
SETTINGS = paths.RUN_SETTINGS[RUN_MODE]
TRACK = paths.track_a_paths(TRACK_A_DIR)
MANIFEST = OUT / "manifests" / "sha_manifest.json"
REGISTRY = OUT / "manifests" / "experiment_registry.jsonl"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if RUN_MODE == "full" and DEVICE != "cuda":
    print("Note: no GPU found. The full run works on CPU but is much slower.")
training.configure_determinism()
training.set_seed(0)
ACTIONS = list(data.ACTION_NAMES)
print("Device:", DEVICE, "| outputs:", OUT)
print("Settings:", SETTINGS)
import itertools
import mi_src.conflict as conflict

Device: cuda | outputs: /content/soccer_mi/.mi_analysis_rerun
Settings: {'reference_epochs': 25, 'batch_windows': 256, 'n_bootstrap': 2000, 'n_draws': 5, 'n_permutations': 1000, 'n_random_circuits': 100, 'probe_train_events': 20000, 'max_events': 100000, 'probe_seeds': 3, 'scrub_draws': 5}


In [3]:
verified_at_start = fingerprint.verify_files(MANIFEST)
manifest_before = {name: entry["sha256"] for name, entry in fingerprint.load_manifest(MANIFEST).items()}
prereg = mi_setup.load_json(OUT / "config" / "preregistration.json")
mi_config = mi_setup.load_json(OUT / "config" / "mi_config.json")
IDENTITY = mi_config["identity_variable"]
assert IDENTITY == "player", "BLOCKED: MB14-MB16 follow up MB04-MB12, which studied the player variable."
PRIMARY_SEED = int(prereg["model"]["primary_seed"])


def run_dir(seed):
    """Run directory of one SEL-4L seed, re-rooted onto this checkout."""
    return paths.rebase_run_dir(prereg["model"]["run_dirs"][str(seed)], TRACK["root"])


SEEDS = [int(seed) for seed in prereg["model"]["run_dirs"]]
N_BOOT = SETTINGS["n_bootstrap"]
BATCH = SETTINGS["batch_windows"]
N_DRAWS = SETTINGS["n_draws"]

events = pd.read_parquet(TRACK["events"])
bundle = mi_setup.build_bundle(events, TRACK, DEVICE)
val_view = bundle["views"]["val"]
# No test view is built or read anywhere in MB14-MB16 (mi_setup.unlock_test_view is never called).
targets = bundle["targets"]
POP = OUT / "artifacts" / "populations"
val_conflicts = pd.read_parquet(POP / "val_conflicts.parquet")
controls = pd.read_parquet(POP / "matched_controls.parquet")
partners = pd.read_parquet(POP / "swap_partners.parquet")
donors = pd.read_parquet(POP / "context_donors.parquet")
partner_of = dict(zip(partners["player"], partners["partner"]))
reference = torch.load(POP / "reference_means.pt", map_location="cpu", weights_only=False)
MEANS = {site: value.to(DEVICE) for site, value in reference["means"].items()}
REFERENCE_WINDOWS = np.asarray(reference["windows"])
MODEL = mi_setup.load_model(run_dir(PRIMARY_SEED), bundle)
N_LAYERS = len(MODEL.blocks)
N_HEADS = MODEL.blocks[0].attention.n_heads
ALL = circuits.all_components(N_LAYERS, N_HEADS)
FEATURES = list(bundle["encoded"].tensors.keys())
IDENTITY_FEATURES = [name for name in ["player", "team", "role", "competition"] if name in FEATURES]
CONTEXT_FEATURES = [name for name in FEATURES if name not in IDENTITY_FEATURES]

p_player = val_conflicts[val_conflicts.population == "P-player"].reset_index(drop=True)
p_player = p_player.merge(donors[["event_row", "donor_window"]], on="event_row", how="left")
discovery = p_player[p_player.half == "discovery"].reset_index(drop=True)
estimation = p_player[p_player.half == "estimation"].reset_index(drop=True)
index0 = val_conflicts[val_conflicts.population == "index-0"].reset_index(drop=True)
print(f"verified files: {len(verified_at_start)} | primary seed {PRIMARY_SEED} | seeds {SEEDS}")
print(f"P-player rows: {len(p_player):,} (discovery {len(discovery):,}, estimation {len(estimation):,}) | "
      f"index-0: {len(index0):,} | matched controls: {len(controls):,}")
print(f"components: {ALL}")
print(f"identity inputs: {IDENTITY_FEATURES} | context inputs ({len(CONTEXT_FEATURES)}): {CONTEXT_FEATURES}")

/content/soccer_mi/model_training_evaluation/src/batching.py:19: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  self.tensors[name] = torch.as_tensor(np.asarray(values), device=device)


verified files: 93 | primary seed 103 | seeds [100, 101, 102, 103, 104]
P-player rows: 1,744 (discovery 1,215, estimation 529) | index-0: 8,947 | matched controls: 252
components: ['L0H0', 'L0H1', 'L0MLP', 'L1H0', 'L1H1', 'L1MLP', 'L2H0', 'L2H1', 'L2MLP', 'L3H0', 'L3H1', 'L3MLP']
identity inputs: ['player', 'team', 'role', 'competition'] | context inputs (15): ['zone', 'period', 'minute_bin', 'score_diff', 'play_pattern', 'poss_len', 'possession_flag', 'prev_action', 'prev_outcome', 'prev_end_zone', 'same_player_as_prev', 'x_norm', 'y_norm', 'dt', 't_poss']


## 1. The plan (written before any number)

In [4]:
PLAN = {
    "status": "follow-up after the deep MI audit; exploratory; not part of the MB12 lock",
    "decision": "the model's top-1 action at the scored position (over all 15 actions); a flip is a change of top-1 between the intact and the edited run",
    "edits": ["player removal", "partner swap", "whole-context swap with all four identity inputs written back"],
    "flip_patching": ("on flip events: denoising (copy one site's intact activation into the edited run; restored if top-1 returns) and "
                      "noising (copy one site's edited activation into the intact run; produced if top-1 changes); sites: each head, each MLP, "
                      "each residual point, at the scored position; rates with match-resampled CIs; estimation half and all validation"),
    "decision_layer": ("logit lens (final LayerNorm and unembedding applied to resid_pre_0 and each resid_post): first layer from which the sign of Delta "
                       "equals the final sign and stays; causal: on flip events, restore all attention and MLP outputs of layers 0..l at the scored "
                       "position (embedding left edited); first l at which the intact top-1 returns"),
    "concentration_reading": "the decision is called concentrated in one layer if one layer holds >= 60 per cent of events' decision layers",
    "self_repair": ("mean-ablate one MLP at the scored position; total effect = change with everything downstream recomputed; direct effect = change with "
                    "every later attention and MLP output frozen at the run's own unablated values; for Delta and the identity effect"),
    "redundancy": "all 6 pairs of MLPs mean-ablated together at the scored position; interaction = joint change - sum of single changes (identity effect and Delta)",
    "cross_model": ("SEL-4L seeds 100-104 and REF-2L, REF-4L (paths from config/reference_models.json): identity-removal and context-swap "
                    "denoising profile at the scored position; C1 mean Delta and share > 0; C2 |partner change| - |random vector change|; "
                    "C3 correlation with the player-only reference's habit change; C4 |I| - |I_floor| with the partner; estimation half")
}
plan_path = OUT / "config" / "MB18_plan.json"
if plan_path.exists():
    stored = mi_setup.load_json(plan_path)
    changed = {key for key in PLAN if key != "written_at" and stored.get(key) != PLAN[key]}
    if changed:
        raise RuntimeError(f"BLOCKED: the plan in {plan_path.name} differs from this notebook in {sorted(changed)}. "
                           "A plan is written once, before results; delete the file only if no result was read.")
    PLAN = stored
    print("plan already written at", stored["written_at"], "- unchanged")
else:
    PLAN["written_at"] = fingerprint.now()
    mi_setup.save_json(plan_path, PLAN)
    print("plan written:", plan_path.name)
fingerprint.record_files(MANIFEST, {"config/MB18_plan.json": plan_path})
print(json.dumps({key: value for key, value in PLAN.items() if key != "written_at"}, indent=1))

plan written: MB18_plan.json
{
 "status": "follow-up after the deep MI audit; exploratory; not part of the MB12 lock",
 "decision": "the model's top-1 action at the scored position (over all 15 actions); a flip is a change of top-1 between the intact and the edited run",
 "edits": [
  "player removal",
  "partner swap",
  "whole-context swap with all four identity inputs written back"
 ],
 "flip_patching": "on flip events: denoising (copy one site's intact activation into the edited run; restored if top-1 returns) and noising (copy one site's edited activation into the intact run; produced if top-1 changes); sites: each head, each MLP, each residual point, at the scored position; rates with match-resampled CIs; estimation half and all validation",
 "decision_layer": "logit lens (final LayerNorm and unembedding applied to resid_pre_0 and each resid_post): first layer from which the sign of Delta equals the final sign and stays; causal: on flip events, restore all attention and MLP outpu

In [5]:
units = p_player[p_player["donor_window"].notna()].copy()
units["donor_window"] = units["donor_window"].astype(int)
units["partner"] = [partner_of.get(p, -1) for p in units["player"]]
units = units[units["partner"] > 0].reset_index(drop=True)
est = units[units.half == "estimation"].reset_index(drop=True)
print(f"units with a donor and a partner: {len(units)} (estimation {len(est)})")
EDIT_NAMES = ["player removal", "partner swap", "whole-context swap"]


def apply_edit(model_inputs, positions, chunk, edit):
    if edit == "player removal":
        return interventions.remove_feature(model_inputs, IDENTITY, positions)
    if edit == "partner swap":
        return interventions.set_feature(model_inputs, "player", positions, torch.as_tensor(chunk["partner"].to_numpy(), device=DEVICE))
    donor, _ = mi_setup.window_inputs(bundle, val_view, chunk["donor_window"].to_numpy())
    return interventions.write_identity(donor, model_inputs, positions, IDENTITY_FEATURES)


def component_sites(n_layers, n_heads):
    """(label, site, head) for every head, MLP and residual point at the scored position."""
    out = [("embedding (resid_pre_0)", "resid_pre_0", None)]
    for l in range(n_layers):
        out += [(f"L{l}H{h}", f"head_z_{l}", h) for h in range(n_heads)]
        out += [(f"L{l}MLP", f"mlp_out_{l}", None), (f"resid_post_{l}", f"resid_post_{l}", None)]
    return out


SITES = component_sites(N_LAYERS, N_HEADS)

units with a donor and a partner: 1716 (estimation 529)


## 2. How often does an edit change the chosen action?

In [6]:
@torch.no_grad()
def top_actions(model, table, edit=None):
    out = np.empty(len(table), dtype=np.int64)
    for part in mi_setup.batch_slices(len(table), BATCH):
        chunk = table.iloc[part]
        inputs, _ = mi_setup.window_inputs(bundle, val_view, chunk["window"].to_numpy())
        positions = torch.as_tensor(chunk["position"].to_numpy(), device=DEVICE)
        if edit is not None:
            inputs = apply_edit(inputs, positions, chunk, edit)
        out[part] = mi_metrics.at_target(hooks.forward(model, inputs)[0], positions).argmax(-1).cpu().numpy()
    return out


TOP = {"intact": top_actions(MODEL, units)}
for edit in EDIT_NAMES:
    TOP[edit] = top_actions(MODEL, units, edit)
choice_rows = []
for label, top in TOP.items():
    choice_rows.append({"run": label, "top-1 = a_ctx": float(np.mean(top == units["a_ctx"].to_numpy())),
                        "top-1 = a_id": float(np.mean(top == units["a_id"].to_numpy())),
                        "top-1 other": float(np.mean((top != units["a_ctx"].to_numpy()) & (top != units["a_id"].to_numpy()))),
                        "changed vs intact": float(np.mean(top != TOP["intact"])) if label != "intact" else 0.0,
                        "changed vs intact (count)": int(np.sum(top != TOP["intact"])) if label != "intact" else 0})
choices = pd.DataFrame(choice_rows)
choices.to_csv(OUT / "results" / "MB18_choices.csv", index=False)
print(choices.round(4).to_string(index=False))

               run  top-1 = a_ctx  top-1 = a_id  top-1 other  changed vs intact  changed vs intact (count)
            intact         0.9289        0.0052       0.0659             0.0000                          0
    player removal         0.9382        0.0012       0.0606             0.0239                         41
      partner swap         0.9336        0.0047       0.0618             0.0332                         57
whole-context swap         0.1737        0.0367       0.7896             0.8205                       1408


## 3. N9 — Which component restores, and which one produces, the change of decision?

On events where an edit changes the top-1 action:

- **Restores** (denoising): the edited run, with one site's activation at the scored position copied from the intact run. Does top-1 go back to the intact action? This tests whether the site is *sufficient* to carry the decision back.
- **Produces** (noising): the intact run, with one site's activation copied from the edited run. Does top-1 change? This tests whether the edit's effect *through that site alone* is enough to change the decision.

Residual points are included as references only: for the player edits, which change one position, patching any residual point at that position restores everything by construction (rate 1). For the context swap, later attention still reads the donor's earlier events, so a residual rate below 1 shows how much the earlier positions matter.

In [7]:
@torch.no_grad()
def flip_patching(model, table, edit, sites):
    rows_out = []
    capture = sorted({site for _, site, _ in sites})
    for part in mi_setup.batch_slices(len(table), BATCH):
        chunk = table.iloc[part]
        inputs, _ = mi_setup.window_inputs(bundle, val_view, chunk["window"].to_numpy())
        positions = torch.as_tensor(chunk["position"].to_numpy(), device=DEVICE)
        edited = apply_edit(inputs, positions, chunk, edit)
        clean_logits, clean_cache = hooks.forward(model, inputs, capture=capture)
        edit_logits, edit_cache = hooks.forward(model, edited, capture=capture)
        clean_top = mi_metrics.at_target(clean_logits, positions).argmax(-1)
        edit_top = mi_metrics.at_target(edit_logits, positions).argmax(-1)
        flipped = (clean_top != edit_top)
        if not flipped.any():
            continue
        frame = {"match_id": chunk["match_id"].to_numpy()[flipped.cpu().numpy()], "half": chunk["half"].to_numpy()[flipped.cpu().numpy()]}
        for label, site, head in sites:
            restored_logits, _ = hooks.forward(model, edited, replace={site: hooks.copy_at(clean_cache[site], positions, head)})
            produced_logits, _ = hooks.forward(model, inputs, replace={site: hooks.copy_at(edit_cache[site], positions, head)})
            restored = mi_metrics.at_target(restored_logits, positions).argmax(-1) == clean_top
            produced = mi_metrics.at_target(produced_logits, positions).argmax(-1) != clean_top
            frame[f"{label}: restores"] = restored[flipped].float().cpu().numpy()
            frame[f"{label}: produces"] = produced[flipped].float().cpu().numpy()
        rows_out.append(pd.DataFrame(frame))
    return pd.concat(rows_out, ignore_index=True) if rows_out else pd.DataFrame()


flip_rows = []
for edit in EDIT_NAMES:
    frame = flip_patching(MODEL, units, edit, SITES)
    for subset in ["estimation", "all validation"]:
        part = frame if subset == "all validation" or len(frame) == 0 else frame[frame.half == "estimation"]
        if len(part) < 10:
            flip_rows.append({"edit": edit, "events": subset, "flips": len(part), "site": "(too few flips; not read)"})
            continue
        for label, _, _ in SITES:
            for kind in ["restores", "produces"]:
                summary = mi_metrics.summarise(part[f"{label}: {kind}"], part["match_id"], N_BOOT)
                flip_rows.append({"edit": edit, "events": subset, "flips": len(part), "site": label, "measure": kind,
                                  "rate": summary["mean"], "CI low": summary["ci_low"], "CI high": summary["ci_high"]})
flips = pd.DataFrame(flip_rows)
flips.to_csv(OUT / "results" / "MB18_flip_patching.csv", index=False)
readable = flips.dropna(subset=["rate"]) if "rate" in flips else flips
if len(readable):
    print(readable.pivot_table(index="site", columns=["edit", "events", "measure"], values="rate")
          .reindex([label for label, _, _ in SITES]).round(3).to_string())
print(flips.drop_duplicates(["edit", "events"])[["edit", "events", "flips"]].to_string(index=False))

edit                      partner swap                              player removal                              whole-context swap                             
events                  all validation          estimation          all validation          estimation              all validation          estimation         
measure                       produces restores   produces restores       produces restores   produces restores           produces restores   produces restores
site                                                                                                                                                           
embedding (resid_pre_0)          1.000    1.000      1.000    1.000          1.000    1.000      1.000    1.000              0.989    0.942      0.984    0.944
L0H0                             0.088    0.070      0.087    0.087          0.073    0.098      0.091    0.091              0.048    0.006      0.049    0.007
L0H1                             0.053  

**How to read it.** A head or MLP with a high *restore* rate carries the decision back by itself; a high *produce* rate means the edit's effect through that site alone changes the decision. A site that restores but does not produce is sufficient but not the only route (others also carry the change). If no single head or MLP restores a meaningful share of flips but residual points do, the decision is spread across components rather than made in one.

## 4. N8 — At which layer is the decision fixed?

**Logit lens.** At each residual point the final LayerNorm and unembedding are applied directly, giving a "Δ so far". The lens decision layer of an event is the first point from which the sign of Δ equals its final sign and stays. This is descriptive (earlier layers were not trained to be read this way).

**Causal.** Patching the residual stream itself is no use here: for an edit at the scored position, restoring the residual at *any* layer restores everything by construction. Instead, on flip events, the intact outputs of **all attention and MLP blocks of layers 0..l** are restored at the scored position while the embedding stays edited. The first l at which the intact action returns is the layer by which the blocks have done the work needed for the decision. If it is never restored, the edited embedding still reaches the output directly (or through earlier positions, for the context swap).

In [8]:
LENS_POINTS = ["resid_pre_0"] + [f"resid_post_{l}" for l in range(N_LAYERS)]


@torch.no_grad()
def lens_deltas(model, table, points):
    out = np.empty((len(table), len(points)))
    for part in mi_setup.batch_slices(len(table), BATCH):
        chunk = table.iloc[part]
        inputs, _ = mi_setup.window_inputs(bundle, val_view, chunk["window"].to_numpy())
        positions = torch.as_tensor(chunk["position"].to_numpy(), device=DEVICE)
        a_ctx = torch.as_tensor(chunk["a_ctx"].to_numpy(), device=DEVICE)
        a_id = torch.as_tensor(chunk["a_id"].to_numpy(), device=DEVICE)
        _, cache = hooks.forward(model, inputs, capture=points)
        rows = torch.arange(len(positions), device=DEVICE)
        for j, point in enumerate(points):
            logits = model.unembed(model.ln_final(cache[point][rows, positions]))
            out[part, j] = mi_metrics.delta(logits, a_ctx, a_id)
    return out


lens = lens_deltas(MODEL, est, LENS_POINTS)
final_sign = np.sign(lens[:, -1])
agrees = np.sign(lens) == final_sign[:, None]
stays = np.flip(np.cumprod(np.flip(agrees, axis=1), axis=1), axis=1).astype(bool)
decision_point = np.argmax(stays, axis=1)
lens_table = pd.DataFrame({"point": LENS_POINTS, "mean lens Delta": lens.mean(0),
                           "share of events whose sign is fixed from here": [float(np.mean(decision_point == j)) for j in range(len(LENS_POINTS))]})
lens_table["mean increase from the previous point"] = np.concatenate([[lens[:, 0].mean()], np.diff(lens, axis=1).mean(0)])
lens_table.to_csv(OUT / "results" / "MB18_logit_lens.csv", index=False)
print(lens_table.round(4).to_string(index=False))
concentrated = lens_table["share of events whose sign is fixed from here"].max() >= 0.6
print(f"lens: largest share at one point {lens_table['share of events whose sign is fixed from here'].max():.3f} "
      f"-> concentrated in one layer (plan rule): {concentrated}")

@torch.no_grad()
def cumulative_restore(model, table, edit):
    """On flip events: restore every attention and MLP output of layers 0..l at the scored position (embedding left edited).
    Returns a [flips, n_layers] array: 1 if top-1 is back to the intact action."""
    n_layers = len(model.blocks)
    sites = [f"{kind}_{l}" for l in range(n_layers) for kind in ["attn_out", "mlp_out"]]
    out, match_ids = [], []
    for part in mi_setup.batch_slices(len(table), BATCH):
        chunk = table.iloc[part]
        inputs, _ = mi_setup.window_inputs(bundle, val_view, chunk["window"].to_numpy())
        positions = torch.as_tensor(chunk["position"].to_numpy(), device=DEVICE)
        edited = apply_edit(inputs, positions, chunk, edit)
        clean_logits, clean_cache = hooks.forward(model, inputs, capture=sites)
        clean_top = mi_metrics.at_target(clean_logits, positions).argmax(-1)
        flipped = mi_metrics.at_target(hooks.forward(model, edited)[0], positions).argmax(-1) != clean_top
        if not flipped.any():
            continue
        restored = []
        for l in range(n_layers):
            replace = {f"{kind}_{k}": hooks.copy_at(clean_cache[f"{kind}_{k}"], positions)
                       for k in range(l + 1) for kind in ["attn_out", "mlp_out"]}
            top = mi_metrics.at_target(hooks.forward(model, edited, replace=replace)[0], positions).argmax(-1)
            restored.append((top == clean_top)[flipped].float().cpu().numpy())
        out.append(np.column_stack(restored))
        match_ids.append(chunk["match_id"].to_numpy()[flipped.cpu().numpy()])
    return (np.concatenate(out), np.concatenate(match_ids)) if out else (np.zeros((0, n_layers)), np.zeros(0))


causal_rows = []
for edit in EDIT_NAMES:
    restored, flip_matches = cumulative_restore(MODEL, est, edit)
    if len(restored) < 10:
        causal_rows.append({"edit": edit, "flips": len(restored), "note": "too few flips"})
        continue
    first = np.where(restored.any(1), np.argmax(restored, axis=1), -1)
    for l in range(N_LAYERS):
        summary = mi_metrics.summarise(restored[:, l], flip_matches, N_BOOT)
        causal_rows.append({"edit": edit, "flips": len(restored), "components restored": f"layers 0..{l}",
                            "share of flips restored": summary["mean"], "CI low": summary["ci_low"], "CI high": summary["ci_high"],
                            "share first restored at this layer": float(np.mean(first == l))})
    causal_rows.append({"edit": edit, "flips": len(restored), "components restored": "never (embedding path needed)",
                        "share first restored at this layer": float(np.mean(first < 0))})
causal_layers = pd.DataFrame(causal_rows)
causal_layers.to_csv(OUT / "results" / "MB18_causal_decision_layer.csv", index=False)
print(causal_layers.round(3).to_string(index=False))

       point  mean lens Delta  share of events whose sign is fixed from here  mean increase from the previous point
 resid_pre_0          -0.3418                                         0.3478                                -0.3418
resid_post_0           1.3585                                         0.4102                                 1.7003
resid_post_1           3.2598                                         0.1966                                 1.9013
resid_post_2           3.9709                                         0.0284                                 0.7111
resid_post_3           5.8937                                         0.0170                                 1.9229
lens: largest share at one point 0.410 -> concentrated in one layer (plan rule): False


              edit  flips           components restored  share of flips restored  CI low  CI high  share first restored at this layer
    player removal     11                   layers 0..0                    0.545   0.091    0.875                               0.545
    player removal     11                   layers 0..1                    0.727   0.300    1.000                               0.182
    player removal     11                   layers 0..2                    0.818   0.500    1.000                               0.091
    player removal     11                   layers 0..3                    0.909   0.700    1.000                               0.091
    player removal     11 never (embedding path needed)                      NaN     NaN      NaN                               0.091
      partner swap     23                   layers 0..0                    0.739   0.583    0.913                               0.739
      partner swap     23                   layers 0..1       

## 5. N7 — Self-repair and redundancy among the MLPs

**Direct vs total.** One MLP is mean-ablated at the scored position. The *total* effect lets every later layer react. The *direct* effect freezes every later attention and MLP output (at every position) at the values of the same run without the ablation, so the change reaches the output only through the residual stream. If total is much smaller than direct, later layers make up for the loss (self-repair), and single-component ablation underestimates the component's role.

**Pairs.** All six pairs of MLPs ablated together. A joint loss larger than the sum of the single losses means the two can stand in for each other (redundancy).

In [9]:
MLP_NAMES = [f"L{l}MLP" for l in range(N_LAYERS)]


@torch.no_grad()
def ablation_runs(model, means, table, ablated, freeze_after=None):
    """(Delta intact-input run, Delta player-removed run) with the given MLPs mean-ablated at the scored position.
    freeze_after=l: every attention and MLP output of layers > l is frozen at the same input's unablated values."""
    out = np.empty((2, len(table)))
    later = [] if freeze_after is None else [f"{kind}_{k}" for k in range(freeze_after + 1, N_LAYERS) for kind in ["attn_out", "mlp_out"]]
    for part in mi_setup.batch_slices(len(table), BATCH):
        chunk = table.iloc[part]
        inputs, _ = mi_setup.window_inputs(bundle, val_view, chunk["window"].to_numpy())
        positions = torch.as_tensor(chunk["position"].to_numpy(), device=DEVICE)
        a_ctx = torch.as_tensor(chunk["a_ctx"].to_numpy(), device=DEVICE)
        a_id = torch.as_tensor(chunk["a_id"].to_numpy(), device=DEVICE)
        for slot, run_inputs in enumerate([inputs, interventions.remove_feature(inputs, IDENTITY, positions)]):
            replace = circuits.ablation_edits(ablated, means, positions) if ablated else {}
            if later:
                _, frozen = hooks.forward(model, run_inputs, capture=later)
                replace = {**replace, **{site: (lambda value, kept=frozen[site]: kept) for site in later}}
            out[slot, part] = circuits.delta_under(model, run_inputs, positions, a_ctx, a_id, replace or None)
    return out


matches = est["match_id"].to_numpy()
intact = ablation_runs(MODEL, MEANS, est, [])
measures = {"Delta": lambda r: r[0], "identity effect": lambda r: r[0] - r[1]}
repair_rows = []
single = {}
for l, mlp in enumerate(MLP_NAMES):
    total = ablation_runs(MODEL, MEANS, est, [mlp])
    direct = ablation_runs(MODEL, MEANS, est, [mlp], freeze_after=l)
    single[mlp] = total
    for name, pick in measures.items():
        total_change, direct_change = pick(total) - pick(intact), pick(direct) - pick(intact)
        t = mi_metrics.summarise(total_change, matches, N_BOOT)
        d = mi_metrics.summarise(direct_change, matches, N_BOOT)
        repair = mi_metrics.summarise(direct_change - total_change, matches, N_BOOT)
        repair_rows.append({"MLP": mlp, "measure": name, "total change": t["mean"], "total CI low": t["ci_low"], "total CI high": t["ci_high"],
                            "direct change": d["mean"], "direct CI low": d["ci_low"], "direct CI high": d["ci_high"],
                            "direct - total (compensation)": repair["mean"], "comp CI low": repair["ci_low"], "comp CI high": repair["ci_high"],
                            "total / direct": t["mean"] / d["mean"] if abs(d["mean"]) > 1e-9 else np.nan})
self_repair = pd.DataFrame(repair_rows)
self_repair.to_csv(OUT / "results" / "MB18_self_repair.csv", index=False)
print(self_repair.round(4).to_string(index=False))

pair_rows = []
for first, second in itertools.combinations(MLP_NAMES, 2):
    joint = ablation_runs(MODEL, MEANS, est, [first, second])
    for name, pick in measures.items():
        joint_change = pick(joint) - pick(intact)
        additive = (pick(single[first]) - pick(intact)) + (pick(single[second]) - pick(intact))
        interaction = mi_metrics.summarise(joint_change - additive, matches, N_BOOT)
        pair_rows.append({"pair": f"{first}+{second}", "measure": name, "joint change": float(joint_change.mean()),
                          "sum of single changes": float(additive.mean()), "joint - sum": interaction["mean"],
                          "CI low": interaction["ci_low"], "CI high": interaction["ci_high"]})
redundancy = pd.DataFrame(pair_rows)
redundancy.to_csv(OUT / "results" / "MB18_redundancy.csv", index=False)
print(redundancy.round(4).to_string(index=False))

  MLP         measure  total change  total CI low  total CI high  direct change  direct CI low  direct CI high  direct - total (compensation)  comp CI low  comp CI high  total / direct
L0MLP           Delta       -7.5110       -8.2291        -6.4731        -0.5484        -0.6056         -0.4644                         6.9625       5.9892        7.6601         13.6955
L0MLP identity effect        0.0954       -0.0109         0.2136        -0.0047        -0.0168          0.0068                        -0.1001      -0.2282        0.0114        -20.3100
L1MLP           Delta       -5.4195       -5.8143        -5.0309        -1.1213        -1.2239         -1.0043                         4.2982       3.8760        4.7276          4.8332
L1MLP identity effect       -0.0669       -0.2116         0.0551        -0.0122        -0.0333          0.0107                         0.0547      -0.0726        0.2012          5.4803
L2MLP           Delta       -4.7115       -5.0250        -4.3361        -1.

       pair         measure  joint change  sum of single changes  joint - sum  CI low  CI high
L0MLP+L1MLP           Delta      -11.4217               -12.9304       1.5087  0.6918   2.2663
L0MLP+L1MLP identity effect        0.1671                 0.0286       0.1385 -0.0116   0.3078
L0MLP+L2MLP           Delta      -11.5440               -12.2224       0.6784  0.0358   1.2593
L0MLP+L2MLP identity effect        0.1285                -0.0067       0.1352 -0.0128   0.2822
L0MLP+L3MLP           Delta      -11.1048               -13.3503       2.2455  1.7440   2.6424
L0MLP+L3MLP identity effect        0.1180                 0.0808       0.0373 -0.0516   0.1075
L1MLP+L2MLP           Delta       -9.9746               -10.1309       0.1564 -0.2845   0.6075
L1MLP+L2MLP identity effect        0.0402                -0.1689       0.2092  0.0665   0.3869
L1MLP+L3MLP           Delta       -9.5405               -11.2588       1.7184  1.4025   2.0174
L1MLP+L3MLP identity effect        0.0565         

**How to read it.** For the identity effect, a change that is more negative than the sum of singles (joint − sum below 0, when the singles reduce the effect) means the two MLPs back each other up. "Direct − total" far from 0 means later layers react to the removal; with the opposite sign of the direct change it is compensation.

## 6. N10 — The same analysis on other models

The site profile and C1–C4 are repeated on every SEL-4L seed and on REF-2L and REF-4L. The reference models' run directories are read from `config/reference_models.json` (a file with `{"REF-2L": "<run dir>", "REF-4L": "<run dir>"}`); if it is missing, only the SEL-4L seeds are run and the notebook says so. Each model's architecture is read from the model itself, so a 2-layer model gets a 2-layer profile. A model that cannot be loaded with this bundle (different encoding or context length) is reported, not forced.

In [10]:
idplayer_state = torch.load(OUT / "artifacts" / "conflict" / "references" / "id_player_seed0.pt", map_location="cpu", weights_only=False)
idplayer = conflict.IdPlayer(idplayer_state["vocabulary_sizes"])
idplayer.load_state_dict(idplayer_state["state"])
with torch.no_grad():
    players_all = torch.arange(idplayer_state["vocabulary_sizes"]["player"])
    HABIT = torch.log_softmax(idplayer(players_all.unsqueeze(1), torch.zeros(len(players_all), 0)) / idplayer_state["temperature"], dim=-1).numpy()

MODELS = {f"SEL-4L seed {seed}": run_dir(seed) for seed in SEEDS}
reference_file = OUT / "config" / "reference_models.json"
if reference_file.exists():
    for name, path in mi_setup.load_json(reference_file).items():
        MODELS[name] = paths.rebase_run_dir(path, TRACK["root"])
else:
    print("config/reference_models.json not found: REF-2L and REF-4L are NOT run. Create it with their run directories to include them.")


def profile_sites(model):
    n_layers = len(model.blocks)
    return ["resid_pre_0"] + [f"{kind}_{l}" for l in range(n_layers) for kind in ["attn_out", "mlp_out", "resid_post"]]


@torch.no_grad()
def model_report(model, table, seed=0):
    generator = torch.Generator().manual_seed(seed)
    sites = profile_sites(model)
    n_layers = len(model.blocks)
    frozen_sites = [f"attn_out_{l}" for l in range(n_layers)] + [f"mlp_out_{l}" for l in range(n_layers)]
    per = {key: [] for key in ["Delta", "removal", "context", "partner change", "random change", "C3 r", "I", "I_floor"]}
    patched = {(contrast, site): [] for contrast in ["removal", "context"] for site in sites}
    for part in mi_setup.batch_slices(len(table), BATCH):
        chunk = table.iloc[part]
        inputs, _ = mi_setup.window_inputs(bundle, val_view, chunk["window"].to_numpy())
        positions = torch.as_tensor(chunk["position"].to_numpy(), device=DEVICE)
        a_ctx = torch.as_tensor(chunk["a_ctx"].to_numpy(), device=DEVICE)
        a_id = torch.as_tensor(chunk["a_id"].to_numpy(), device=DEVICE)
        d = lambda logits: mi_metrics.delta(mi_metrics.at_target(logits, positions), a_ctx, a_id)
        clean_logits, clean_cache = hooks.forward(model, inputs, capture=sites + frozen_sites)
        base = d(clean_logits)
        per["Delta"].append(base)
        partner = torch.as_tensor(chunk["partner"].to_numpy(), device=DEVICE)
        for contrast, edit in [("removal", "player removal"), ("context", "whole-context swap")]:
            corrupt = apply_edit(inputs, positions, chunk, edit)
            corrupt_value = d(hooks.forward(model, corrupt)[0])
            per[contrast].append(np.stack([base, corrupt_value]))
            for site in sites:
                patched[(contrast, site)].append(d(hooks.forward(model, corrupt, replace={site: hooks.copy_at(clean_cache[site], positions)})[0]))
        swapped_logits = hooks.forward(model, interventions.set_feature(inputs, "player", positions, partner))[0]
        per["partner change"].append(d(swapped_logits) - base)
        changes = [np.abs(d(hooks.forward(model, inputs, replace={"factor_player": interventions.norm_matched_vector(None, positions, generator)})[0]) - base)
                   for _ in range(N_DRAWS)]
        per["random change"].append(np.mean(changes, 0))
        model_change = mi_metrics.log_softmax(mi_metrics.at_target(swapped_logits, positions).cpu().numpy()) - \
            mi_metrics.log_softmax(mi_metrics.at_target(clean_logits, positions).cpu().numpy())
        model_change -= model_change.mean(1, keepdims=True)
        habit_change = HABIT[chunk["partner"].to_numpy()] - HABIT[chunk["player"].to_numpy()]
        habit_change -= habit_change.mean(1, keepdims=True)
        per["C3 r"].append((model_change * habit_change).sum(1) / (np.sqrt((model_change ** 2).sum(1) * (habit_change ** 2).sum(1)) + 1e-12))
        donor, _ = mi_setup.window_inputs(bundle, val_view, chunk["donor_window"].to_numpy())
        cells, floor = {}, {}
        for name, context in [("X", inputs), ("Y", donor)]:
            with_a = interventions.set_feature(context, "player", positions, torch.as_tensor(chunk["player"].to_numpy(), device=DEVICE))
            logits_a, cache_a = hooks.forward(model, with_a, capture=frozen_sites)
            with_b = interventions.set_feature(context, "player", positions, partner)
            cells[("A", name)], cells[("B", name)] = d(logits_a), d(hooks.forward(model, with_b)[0])
            floor[name] = d(hooks.forward(model, with_b, replace={s: (lambda v, kept=cache_a[s]: kept) for s in frozen_sites})[0])
        per["I"].append(cells[("A", "X")] - cells[("A", "Y")] - cells[("B", "X")] + cells[("B", "Y")])
        per["I_floor"].append(cells[("A", "X")] - cells[("A", "Y")] - floor["X"] + floor["Y"])
    per = {key: np.concatenate(values, axis=-1) for key, values in per.items()}
    patched = {key: np.concatenate(values) for key, values in patched.items()}
    m = table["match_id"].to_numpy()
    behaviour = {"C1 mean Delta": mi_metrics.summarise(per["Delta"], m, N_BOOT),
                 "C1 share Delta > 0": mi_metrics.summarise((per["Delta"] > 0).astype(float), m, N_BOOT),
                 "identity removal effect": mi_metrics.summarise(per["removal"][0] - per["removal"][1], m, N_BOOT),
                 "C2 |partner| - |random vector|": mi_metrics.summarise(np.abs(per["partner change"]) - per["random change"], m, N_BOOT),
                 "C3 mean r": mi_metrics.summarise(per["C3 r"], m, N_BOOT),
                 "C4 |I| - |I_floor|": mi_metrics.summarise(np.abs(per["I"]) - np.abs(per["I_floor"]), m, N_BOOT)}
    profile = []
    for (contrast, site), values in patched.items():
        clean, corrupt = per[contrast][0], per[contrast][1]
        result = mi_metrics.pooled_recovery(values - corrupt, clean - corrupt, m, N_BOOT)
        profile.append({"contrast": contrast, "site": site, "recovery": result["recovery"], "CI low": result["ci_low"], "CI high": result["ci_high"]})
    return behaviour, pd.DataFrame(profile)


behaviour_rows, profile_frames = [], []
for name, directory in MODELS.items():
    try:
        model = MODEL if name == f"SEL-4L seed {PRIMARY_SEED}" else mi_setup.load_model(directory, bundle)
    except Exception as error:
        behaviour_rows.append({"model": name, "measure": "(not loaded)", "mean": np.nan, "note": f"{type(error).__name__}: {error}"[:200]})
        continue
    behaviour, profile = model_report(model, est)
    architecture = f"{len(model.blocks)} layers x {model.blocks[0].attention.n_heads} heads"
    for measure, summary in behaviour.items():
        behaviour_rows.append({"model": name, "architecture": architecture, "measure": measure, "mean": summary["mean"],
                               "CI low": summary["ci_low"], "CI high": summary["ci_high"]})
    profile_frames.append(profile.assign(model=name, architecture=architecture))
    if model is not MODEL:
        del model
cross_behaviour = pd.DataFrame(behaviour_rows)
cross_profile = pd.concat(profile_frames, ignore_index=True)
cross_behaviour.to_csv(OUT / "results" / "MB18_cross_model_behaviour.csv", index=False)
cross_profile.to_csv(OUT / "results" / "MB18_cross_model_profile.csv", index=False)
print(cross_behaviour.pivot_table(index="measure", columns="model", values="mean").round(4).to_string())
print()
for contrast in ["removal", "context"]:
    print(f"denoising profile, {contrast} (recovery at the scored position):")
    print(cross_profile[cross_profile.contrast == contrast].pivot_table(index="site", columns="model", values="recovery").round(3).to_string())

model                           REF-2L  REF-4L  SEL-4L seed 100  SEL-4L seed 101  SEL-4L seed 102  SEL-4L seed 103  SEL-4L seed 104
measure                                                                                                                            
C1 mean Delta                   5.9831  6.1982           6.0161           5.9573           6.2468           5.8937           6.4042
C1 share Delta > 0              0.9830  0.9849           0.9830           0.9849           0.9754           0.9811           0.9754
C2 |partner| - |random vector| -0.2009 -0.1207          -0.0847           0.0041          -0.0694          -0.0841          -0.0612
C3 mean r                       0.0806  0.0578           0.0595           0.0890           0.0851           0.0758           0.0918
C4 |I| - |I_floor|              0.8435  0.8382           0.7204           0.7595           0.9344           0.7869           0.7986
identity removal effect        -0.0597 -0.1227          -0.1668          -0.

**How to read it.** The four SEL-4L replicate seeds test whether the profile is a property of this training recipe or of one run. REF-4L has the same depth and a different training setting; REF-2L has half the depth. If the identity and context effects run through the MLPs at the scored position in all of them, "a shared MLP chain at the scored event" is a general finding for this architecture family. If REF-2L's profile is different, the mechanism depends on depth.

## 7. Summary and integrity

In [11]:
summary = {"status": PLAN["status"], "choices": choices.to_dict(orient="records"), "flip_patching": flips.to_dict(orient="records"),
           "logit_lens": lens_table.to_dict(orient="records"), "causal_decision_layer": causal_layers.to_dict(orient="records"),
           "self_repair": self_repair.to_dict(orient="records"), "redundancy": redundancy.to_dict(orient="records"),
           "cross_model_behaviour": cross_behaviour.to_dict(orient="records"), "cross_model_profile": cross_profile.to_dict(orient="records"),
           "run_mode": RUN_MODE}
mi_setup.save_json(OUT / "results" / "MB18_summary.json", summary)
fingerprint.record_files(MANIFEST, {"results/MB18_summary.json": OUT / "results" / "MB18_summary.json"})
fingerprint.register(REGISTRY, {"notebook": "MB18", "experiment": "decision flips, decision layer, self-repair, redundancy, cross-model replication",
                                "seed": PRIMARY_SEED, "run_mode": RUN_MODE})

In [12]:
verified_at_end = fingerprint.verify_files(MANIFEST)
manifest_after = {name: entry["sha256"] for name, entry in fingerprint.load_manifest(MANIFEST).items()}
changed = sorted(name for name in manifest_before if manifest_after.get(name) != manifest_before[name])
print(f"manifest entries verified at the end: {len(verified_at_end)}")
print("earlier entries whose fingerprint changed during this notebook:", changed if changed else "none")
assert not changed, "BLOCKED: an earlier Track B file changed while this notebook ran."
test_log = OUT / "logs" / "test_access_log.jsonl"
test_lines = len(test_log.read_text().strip().splitlines()) if test_log.exists() and test_log.read_text().strip() else 0
print(f"test accesses recorded so far: {test_lines} (MB12 recorded exactly one; this notebook adds none)")

manifest entries verified at the end: 95
earlier entries whose fingerprint changed during this notebook: none
test accesses recorded so far: 1 (MB12 recorded exactly one; this notebook adds none)


**Researcher interpretation** *(written by the researcher after reading the numbers above; the code never writes these words)*

- What was tested:
- What was observed:
- What it supports / contradicts:
- What it does not establish:
- Evidence label: